# M2 — First Backtest

End-to-end demo of the M2 backtest engine: load BTC/USDT daily bars from the local DuckDB store, run two strategies, compare equity curves and metrics.

If you see no data, run `uv run hindcast sync` first.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from hindcast.backtest.engine import BacktestEngine
from hindcast.backtest.execution import SimpleExecutionModel
from hindcast.backtest.portfolio import Portfolio
from hindcast.backtest.strategies.buy_and_hold import BuyAndHold
from hindcast.backtest.strategies.ma_crossover import MACrossover
from hindcast.backtest.types import Bar
from hindcast.config import settings
from hindcast.data.storage import Storage

storage = Storage(settings.db_path)
df = storage.query_ohlcv(
    'binance', 'BTC/USDT', '1d',
    start=pd.Timestamp('2024-01-01', tz='UTC'),
    end=pd.Timestamp('2026-01-01', tz='UTC'),
)
bars = [Bar.from_series(row) for _, row in df.iterrows()]
print(f'Loaded {len(bars)} bars: {bars[0].timestamp.date()} to {bars[-1].timestamp.date()}')

## Run two strategies

Same data, same fees and slippage. The only thing that changes is the decision rule.

In [ ]:
INITIAL_CASH = 10_000.0

def run(label, strategy):
    engine = BacktestEngine(
        strategy=strategy,
        execution_model=SimpleExecutionModel(fee_pct=0.001, slippage_pct=0.0005),
        portfolio=Portfolio(initial_cash=INITIAL_CASH),
    )
    return label, engine.run(bars)

results = dict([
    run('buy_and_hold', BuyAndHold(allocation_pct=0.99)),
    run('ma_crossover (5/20)', MACrossover(fast_window=5, slow_window=20, allocation_pct=0.99)),
])

rows = []
for label, res in results.items():
    m = res.metrics
    rows.append({
        'strategy': label,
        'final': res.equity_curve['equity'].iloc[-1],
        'total_return': m.total_return,
        'annualized': m.annualized_return,
        'max_dd': m.max_drawdown,
        'sharpe': m.sharpe_ratio,
        'trades': m.n_trades,
        'win_rate': m.win_rate,
        'profit_factor': m.profit_factor,
    })
pd.DataFrame(rows).set_index('strategy')

## Equity curves overlaid

Visualizing both side-by-side makes the trade-off concrete: the active strategy reduces drawdown but at the cost of giving up most of the upside.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
for label, res in results.items():
    ax.plot(res.equity_curve['timestamp'], res.equity_curve['equity'], label=label, linewidth=1.5)
ax.axhline(INITIAL_CASH, color='gray', linestyle='--', alpha=0.5, label='initial cash')
ax.set_xlabel('Date')
ax.set_ylabel('Equity ($)')
ax.set_title('BTC/USDT 1d — equity comparison')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Takeaway

Buy-and-hold is a stubborn baseline in a strong-trend market. M3 will reproduce a few more strategies and try to find one that genuinely earns its complexity — anything that doesn't is just paying fees to feel busy.